# Import FRED data
- Author: Bryan Bravo
- Created: 2026-03-02
## Import Libraries

In [ ]:
import os 
import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

### Custom Functions

In [32]:
def fetch_fred_data(
    api_key: str,
    currency_name: str, # Currency name of series id.
    series_id: str,  # FRED series ID
    rate_name: str = None,
    start_date: str = "1980-01-01",
    end_date: str = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
    ) -> pd.DataFrame:
    """
    Fetches daily foreign exchange rate data from the FRED API, cleans it, and returns
    a standardized DataFrame containing the latest revision for each date.

    This function:
    - Adds a `currency` column for identification
    - Computes a unified USD exchange rate (`us_fx_rate`) so that:
        • If the series is FX-per-USD (e.g., DEXJPUS), it inverts the value
        • Otherwise, it uses the value as-is

    Parameters
    ----------
    api_key : str
        FRED API key used for authentication.
    currency_name : str
        Human-readable currency name to attach to the output (e.g., "euro", "yen").
    series_id : str
        FRED series ID (e.g., "DEXUSEU", "DEXJPUS").
    rate_name : str
        If API value is based on interest rates, this will create a column name matching the string value.
    start_date : str
        Start date for the query in YYYY-MM-DD format. (Defaults to "1980-01-01")
    end_date : str
        End date for the query in YYYY-MM-DD format. (Defaults to the last day of previous month)

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - date : datetime64
        - currency : str
        - `rate_name` <- value derived from variable given : float
        representing the latest available FX rate for each date.

    Raises
    ------
    Exception
        If the API request fails or the response cannot be parsed.

    """

    try:
        response = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&realtime_start={start_date}&realtime_end={end_date}&api_key={api_key}&file_type=json",
            timeout=10)
        fred_data = response.json()

        if 'error_code' in fred_data:  # check if response is None
            print(f"✗ FRED API error: {fred_data.get('error_code', 'error_message')}")

        # Create pandas DataFrame from observations
        df = pd.DataFrame(fred_data['observations'])

        # Select relevant columns
        df = df[['date', 'realtime_start', 'value']]
        df['currency'] = currency_name


        # Correct data types
        df['date'] = pd.to_datetime(df['date'])
        df['realtime_start'] = pd.to_datetime(df['realtime_start'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Convert to numeric, coerce errors to NaN

        # Add USD exchange rate (USD/FX)
        if rate_name == 'fx_rate':
            if series_id[5:] == "US":
                df['fx_rate'] = 1 / df['value']
            else:
                df['fx_rate'] = df['value']

            # Keep only the most recent revision for each date and subset columns
            df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
            df = df[['date', 'currency', 'fx_rate']].reset_index(drop=True)

        # Add column with identified rate name
        else:
            df[rate_name] = df['value']
            # Keep only the most recent revision for each date and subset columns
            df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
            df = df[['date', 'currency', rate_name]].reset_index(drop=True)
        

        print(f"✓ Fetched {len(df)} records for '{currency_name}' from FRED for years {dt.strftime(df['date'].min(), '%Y-%m-%d')} through {dt.strftime(df['date'].max(), '%Y-%m-%d')}")
        return df
    except Exception as e:
        print(f"✗ Error fetching FRED data on {currency_name}: {str(e)[:100]}")

### Variables

In [33]:
fred_api = hardcoded_keys.FRED_API_KEY

## Query

In [47]:
fred_fx_series = {
    "euro": {
        'interest_rate_24h': 'IRSTCI01EZM156N',
        'interest_rate_90d': 'IR3TIB01EZM156N',
        'interest_rate_10y': 'IRLTLT01EZM156N',
        # 'fx_rate': 'DEXUSEU'
    },
    "japan": {
        'interest_rate_24h': 'IRSTCI01JPM156N',
        'interest_rate_90d': 'IR3TIB01JPM156N',
        'interest_rate_10y': 'IRLTLT01JPM156N',
        # 'fx_rate': 'DEXJPUS'
    },
    # "china": "DEXCHUS",
    # "canada": "DEXCAUS",
    # "united_kingdom": "DEXUSUK",
    # "south_korea": "DEXKOUS",
    # "mexico": "DEXMXUS",
    # "india": "DEXINUS",
    # "brazil": "DEXBZUS",
    # "australia": "DEXUSAL",
    # "switzerland": "DEXSZUS",
    # "sweden": "DEXSDUS",
    # "norway": "DEXNOUS",
    # "denmark": "DEXDNUS",
    # "hong_kong": "DEXHKUS",
    # "singapore": "DEXSIUS",
    # "south_africa": "DEXSFUS",
    # "taiwan": "DEXTAUS",
    # "thailand": "DEXTHUS",
    # "new_zealand": "DEXUSNZ",
    # "malaysia": "DEXMAUS",
    # "venezuela": "DEXVZUS",
    # "united_states": "DTWEXBGS"  # Nominal Broud U.S. Dollar Index
}


for country, kval in fred_fx_series.items():
    fred_df = pd.DataFrame([[]])

    for i, (rate_name, series) in enumerate(kval.items()):
        if i == 0:
            df1 = fetch_fred_data(fred_api, country, series, rate_name)
        
        df2 = fetch_fred_data(fred_api, country, series, rate_name)
        df1 = df1.merge(df2, how='outer', on=[df1['date'].dt.year == df2['date'].dt.year,
                                             df1['date'].dt.month == df2['date'].dt.month])

    fred_df = pd.concat([fred_df, df1], ignore_index=True)

fred_df

✓ Fetched 385 records for 'euro' from FRED for years 1994-01-01 through 2026-01-01
✓ Fetched 385 records for 'euro' from FRED for years 1994-01-01 through 2026-01-01
✓ Fetched 385 records for 'euro' from FRED for years 1994-01-01 through 2026-01-01


KeyError: 'date'

In [ ]:
# fred_fx_series = {
#     "euro": "DEXUSEU",
#     "japan": "DEXJPUS",
#     "china": "DEXCHUS",
#     "canada": "DEXCAUS",
#     "united_kingdom": "DEXUSUK",
#     "south_korea": "DEXKOUS",
#     "mexico": "DEXMXUS",
#     "india": "DEXINUS",
#     "brazil": "DEXBZUS",
#     "australia": "DEXUSAL",
#     "switzerland": "DEXSZUS",
#     "sweden": "DEXSDUS",
#     "norway": "DEXNOUS",
#     "denmark": "DEXDNUS",
#     "hong_kong": "DEXHKUS",
#     "singapore": "DEXSIUS",
#     "south_africa": "DEXSFUS",
#     "taiwan": "DEXTAUS",
#     "thailand": "DEXTHUS",
#     "new_zealand": "DEXUSNZ",
#     "malaysia": "DEXMAUS",
#     "venezuela": "DEXVZUS",
#     "united_states": "DTWEXBGS"  # Nominal Broud U.S. Dollar Index
# }

# fred_data_df = reduce(lambda x, y: pd.concat([x, y], ignore_index=True),
#                       [fetch_fred_data(hardcoded_keys.FRED_API_KEY, cur, ser) for cur, ser in fred_fx_series.items()])
# fred_data_df.sort_values(['currency', 'date'], ascending=[True, True])


✓ Fetched 7080 currency records for 'euro' from FRED for years 1999-01-04 through 2026-02-20
✓ Fetched 14385 currency records for 'japan' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 11776 currency records for 'china' from FRED for years 1981-01-02 through 2026-02-20
✓ Fetched 14385 currency records for 'canada' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 14385 currency records for 'united_kingdom' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 11705 currency records for 'south_korea' from FRED for years 1981-04-13 through 2026-02-20
✓ Fetched 8425 currency records for 'mexico' from FRED for years 1993-11-08 through 2026-02-20
✓ Fetched 13864 currency records for 'india' from FRED for years 1973-01-02 through 2026-02-20
✓ Fetched 8125 currency records for 'brazil' from FRED for years 1995-01-02 through 2026-02-20
✓ Fetched 14385 currency records for 'australia' from FRED for years 1971-01-04 through 2026-02-20
✓ Fetched 14385 currency records

,date,currency,fx_rate
104130,1971-01-04,australia,1.112700
104131,1971-01-05,australia,1.113200
104132,1971-01-06,australia,1.114000
104133,1971-01-07,australia,1.113800
104134,1971-01-08,australia,1.112400
...,...,...,...
273718,2026-02-16,venezuela,NaN
273719,2026-02-17,venezuela,0.002546
273720,2026-02-18,venezuela,0.002546
273721,2026-02-19,venezuela,0.002514


In [35]:
api_key = hardcoded_keys.FRED_API_KEY
currency_name = 'euro' # Currency name of series id.
series_id = 'IRSTCI01EZM156N'  # FRED series ID
interest_rate_name = 'intrst_rt_24h'
start_date = "1980-01-01"
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")

response = requests.get(
    f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&realtime_start={start_date}&realtime_end={end_date}&api_key={api_key}&file_type=json",
    timeout=10)
fred_data = response.json()

if 'error_code' in fred_data:  # check if response is None
    print(f"✗ FRED API error: {fred_data.get('error_code', 'error_message')}")

# Create pandas DataFrame from observations
df = pd.DataFrame(fred_data['observations'])

# Select relevant columns
df = df[['date', 'realtime_start', 'value']]
df['currency'] = currency_name


# Correct data types
df['date'] = pd.to_datetime(df['date'])
df['realtime_start'] = pd.to_datetime(df['realtime_start'])
df['value'] = pd.to_numeric(df['value'], errors='coerce')  # Convert to numeric, coerce errors to NaN

if interest_rate_name:
    df[interest_rate_name] = df['value']
    # Keep only the most recent revision for each date and subset columns
    df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
    df = df[['date', 'currency', interest_rate_name]].reset_index(drop=True)

# Add USD exchange rate (USD/FX)
else:
    if series_id[5:] == "US":
        df['fx_rate'] = 1 / df['value']
    else:
        df['fx_rate'] = df['value']

    # Keep only the most recent revision for each date and subset columns
    df = df.loc[df.groupby('date')['realtime_start'].idxmax()]
    df = df[['date', 'currency', 'fx_rate']].reset_index(drop=True)

print(f"✓ Fetched {len(df)} currency records for '{currency_name}' from FRED for years {dt.strftime(df['date'].min(), '%Y-%m-%d')} through {dt.strftime(df['date'].max(), '%Y-%m-%d')}")
df['date'].dt.year

✓ Fetched 385 currency records for 'euro' from FRED for years 1994-01-01 through 2026-01-01


0      1994
1      1994
2      1994
3      1994
4      1994
       ... 
380    2025
381    2025
382    2025
383    2025
384    2026
Name: date, Length: 385, dtype: int32

In [ ]:
fred_data_df.info()
fred_data_df.value_counts('currency').sort_index()